# Retrieval-Augmented Generation (RAG)

In [1]:
from setup import bedrock_tool
import chromadb
from pathlib import Path
from agents import Agent, Runner, function_tool, trace

✅ AWS credentials found
✅ OpenAI credentials found
✅ EXA credentials found


Create a static calorie table that we can use as a tool:

In [2]:
# We populated the RAG with the data from the data/calories.csv file in
# the ../rag_setup/rag_setup.ipynb notebook

chroma_client = chromadb.PersistentClient("../chroma")
nutrition_db = chroma_client.get_collection(name="nutrition_db")

In [3]:
results = nutrition_db.query(query_texts=["banana"], n_results=2)
for i, doc in enumerate(results["documents"][0]):   
    print(doc)
    print("\n")

Food: Banana
        Category: Fruits
        Nutritional Information:
        - Calories: 89 per 100g
        - Energy: 374 kJ per 100g
        - Serving size reference: 100g

        This is a fruits food item that provides 89 calories per 100 grams.


Food: Banana Juice
        Category: (Fruit)Juices
        Nutritional Information:
        - Calories: 50 per 100g
        - Energy: 210 kJ per 100g
        - Serving size reference: 100ml

        This is a (fruit)juices food item that provides 50 calories per 100 grams.




In [4]:
@function_tool   # openai agent compatible
def calorie_lookup_tool(query: str, max_results: int = 3) -> str:
    """
    Tool function for a RAG database to look up calorie information for specific food items, but not for meals.

    Args:
        query: The food item to look up.
        max_results: The maximum number of results to return.

    Returns:
        A string containing the nutrition information.
    """

    results = nutrition_db.query(query_texts=[query], n_results=max_results)

    if not results["documents"][0]:
        return f"No nutrition information found for: {query}"

    # Format results for the agent
    formatted_results = []
    for i, doc in enumerate(results["documents"][0]):
        metadata = results["metadatas"][0][i]
        food_item = metadata["food_item"].title()
        calories = metadata["calories_per_100g"]
        category = metadata["food_category"].title()

        formatted_results.append(
            f"{food_item} ({category}): {calories} calories per 100g"
        )

    return "Nutrition Information:\n" + "\n".join(formatted_results)

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `calorie_lookup_tool` function_

In [ ]:
# print(calorie_lookup_tool('bananas'))

In [5]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    If you need to look up calorie information, use the calorie_lookup_tool.
    """,
    model="litellm/bedrock/eu.amazon.nova-lite-v1:0",
    tools=[bedrock_tool(calorie_lookup_tool.__dict__)],
)

In [6]:
with trace("Nutrition Assistant with RAG") as t:
    result = await Runner.run(
        calorie_agent,
        "How many calories are in total in a banana and an apple? Also give calories per 100g",
    )

print(f"Output: {result.final_output}")
print(f"\nTrace URL: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")

Output: Total calories for a banana and an apple:

- Average banana weight: 118g
- Average apple weight: 182g

Banana: 89 calories per 100g
Apple: 52 calories per 100g

Banana (118g): 105.02 calories
Apple (182g): 94.64 calories

Total calories: 105.02 + 94.64 = 199.66 calories

Calories per 100g:
- Banana: 89 calories
- Apple: 52 calories

Trace URL: https://platform.openai.com/logs/trace?trace_id=trace_f0fc4a6a80a7480393d4eb0e6dec710c


## Another example with a different dataset
- Make this notebook answer health-related nutritional questions
- Use this collection in chromadb: `nutrition_qna`

In [ ]:
chroma_client = chromadb.PersistentClient("../chroma")
nutrition_db = chroma_client.get_collection(name="nutrition_qna")

In [36]:
@function_tool 
def nutrition_qna_tool(query: str) -> str:
    """
    Tool function for a RAG database to look up answers to nutrition and health related questions.

    Args:
        query: The question to look up.

    Returns:
        A string containing the answer to the question.
    """
    results = nutrition_db.query(query_texts=[query], n_results=1)

    if not results["documents"][0]:
        return f"No information found for: {query}"
    
    return results["documents"][0][0]

In [35]:
# nutrition_qna_tool('Why is Vitamin C important for your health?')

In [37]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition and health assistant answering questions related to health and nutrition.
    You give concise answers.
    If you need to look up answers, use the nutrition_qna_tool.
    """,
    model="litellm/bedrock/eu.amazon.nova-lite-v1:0",
    tools=[bedrock_tool(nutrition_qna_tool.__dict__)],
)

In [38]:
with trace("Nutrition Assistant Q&A") as t:
    result = await Runner.run(
        calorie_agent,
        "Why is Vitamin C important for your health?",
    )

print(f"Output: {result.final_output}")
print(f"\nTrace URL: https://platform.openai.com/logs/trace?trace_id={t.trace_id}")

Output: 
Vitamin C is important for your health as it aids in red blood cell formation and metabolism, helps with iron absorption, and supports collagen production. You can find this essential vitamin in foods like citrus fruits, kiwi, tomatoes, melons, strawberries, dark leafy vegetables, chili peppers, cabbage, and broccoli.

Trace URL: https://platform.openai.com/logs/trace?trace_id=trace_523ec0aac98c4eb89f9f086dd3513c94
